# 🔍 CausalLens — IBM HR Attrition Analysis

**Causal Question**: Does earning above-median income causally *reduce* the probability
of employee attrition, after controlling for job level, satisfaction, and tenure?

| Variable | Role | Description |
|----------|------|-------------|
| `HighIncome` | Treatment | 1 = above-median monthly income |
| `Attrition_num` | Outcome | 1 = employee left, 0 = stayed |
| Age, JobLevel, Satisfaction, OverTime, … | Confounders | HR features |

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
print('Libraries loaded ✓')

## 1. Load & Explore Data

In [ ]:
from data.load_data import load_ibm, dataset_summary, IBM_TREATMENT, IBM_OUTCOME, IBM_CONFOUNDERS

df = load_ibm()
print(f'Shape: {df.shape}')
print(f'Attrition rate: {df[IBM_OUTCOME].mean():.1%}')
print(f'High-income rate: {df[IBM_TREATMENT].mean():.1%}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Attrition rate by income group
attr_by_inc = df.groupby(IBM_TREATMENT)[IBM_OUTCOME].mean().reset_index()
attr_by_inc[IBM_TREATMENT] = attr_by_inc[IBM_TREATMENT].map({0: 'Low Income', 1: 'High Income'})
axes[0].bar(attr_by_inc[IBM_TREATMENT], attr_by_inc[IBM_OUTCOME],
            color=['#e74c3c','#2ecc71'], edgecolor='white', width=0.5)
axes[0].set_ylabel('Attrition Rate')
axes[0].set_title('Attrition Rate by Income Group')

# Job satisfaction distribution
for grp, lbl, c in [(0,'Low Income','#e74c3c'),(1,'High Income','#2ecc71')]:
    axes[1].hist(df.loc[df[IBM_TREATMENT]==grp,'JobSatisfaction'],
                 bins=4, alpha=0.6, label=lbl, color=c, edgecolor='white')
axes[1].set_title('Job Satisfaction by Income Group')
axes[1].legend()

# Years at company
df.boxplot(column='YearsAtCompany', by=IBM_TREATMENT, ax=axes[2])
axes[2].set_title('Years at Company by Income Group')
axes[2].set_xlabel('HighIncome')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 2. Causal DAG

In [ ]:
from causal.discovery import build_causal_graph, plot_causal_graph

G = build_causal_graph('ibm', IBM_TREATMENT, IBM_OUTCOME)
fig = plot_causal_graph(G, IBM_TREATMENT, IBM_OUTCOME,
                        title='Causal DAG — IBM HR Attrition')
plt.show()

## 3. Causal Effect Estimation

In [ ]:
from causal.effect_estimation import estimate_all_effects

results = estimate_all_effects(
    df,
    treatment=IBM_TREATMENT,
    outcome=IBM_OUTCOME,
    confounders=IBM_CONFOUNDERS,
    dataset_name='IBM HR Attrition',
    graph=G,
)

print(f'\nNaive ATE (mean diff)  : {results.naive_ate:+.4f}')
print(f'PSM ATE                : {results.psm_ate:+.4f}')
print(f'IPW ATE                : {results.ipw_ate:+.4f}')
print(f'Double ML ATE          : {results.dml_ate:+.4f}  95% CI [{results.dml_ci_lower:.4f}, {results.dml_ate_upper:.4f}]')
print(f'Causal Forest ATE      : {results.cfdml_ate:+.4f}  95% CI [{results.cfdml_ci_lower:.4f}, {results.cfdml_ci_upper:.4f}]')
print('\nInterpretation: a negative ATE means high income REDUCES attrition probability.')

In [ ]:
summary_df = results.summary_df()
display(summary_df)

fig, ax = plt.subplots(figsize=(8, 3.5))
ates = summary_df['ATE'].astype(float)
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in ates]
bars = ax.barh(summary_df['Method'], ates, color=colors, height=0.5)
ax.axvline(0, color='#333', lw=1.2, ls='--')
ax.axvline(results.naive_ate, color='#f39c12', lw=1.5, ls=':',
           label=f'Naive ATE ({results.naive_ate:+.4f})')
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=9)
ax.set_xlabel('ATE (change in P(Attrition))')
ax.set_title('Causal vs Naive ATE — IBM HR Attrition', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Refutation Tests

In [ ]:
from causal.refutation import run_all_refutations, refutations_to_df

refutations = run_all_refutations(
    df, IBM_TREATMENT, IBM_OUTCOME, IBM_CONFOUNDERS,
    original_ate=results.dml_ate or results.naive_ate,
    n_simulations=100,
)

display(refutations_to_df(refutations))
for r in refutations:
    icon = '✅' if r.passed else '⚠️'
    print(f'{icon} {r.test_name}: {r.interpretation}')

## 5. LLM Business Report

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

from llm.report_generator import generate_full_report

dataset_desc = (
    'IBM HR Attrition dataset (n=1470). '
    'We investigate whether earning above-median income causally reduces '
    'employee attrition probability, controlling for job level, satisfaction, '
    'overtime, and tenure.'
)

report = generate_full_report(results, refutations, dataset_desc)
print(report)